# Surgical Patient Flow Intelligence

- [Architecture](#architecture)
- [0. Environment & Imports](#environment-imports)
- [Database Connection](#database-connection)
  - [Check data of each table](#check-data-each-table)
  - [Check Columns of Each Table](#check-columns-each-table)
- [2. Basic Statistics](#basic-statistics)
  - [Missing Value Percentage](#missing-value-percentage)
- [3. Exploratory Data Analysis](#exploratory-data-analysis)
- [Merge Tables](#merge-tables)
  - [Date Conversion](#date-conversion)
  - [Join Operational Table](#join-operational-table)
  - [Feature Engineering](#feature-engineering)
  - [Identify High Cardinality Features](#high-cardinality-features)
  - [Encoding](#encoding)
  - [Fill Missing Values](#fill-missing-values)
- [Model Building](#model-building)
  - [MODEL 1 - DISCHARGE READINESS](#model-1-discharge-readiness)
  - [Calibrated Model](#calibrated-model)
  - [Prediction & Evaluation](#prediction-evaluation)
  - [Precision-Recall Analysis](#precision-recall-analysis)
  - [FINAL Decision Logic](#final-decision-logic)
  - [Final Output & Analysis](#final-output-analysis)
  - [SHAP Explanation for Discharge Readiness](#shap-explanation-discharge-readiness)
- [MODEL 2 - LOS Prediction](#model-2-los-prediction)
  - [1. Admission LOS Model](#admission-los-model)
    - [Train Model](#train-model-admission)
    - [Evaluation](#evaluation-admission)
  - [PART 2 POST-OPERATIVE MODEL](#part-2-post-operative-model)
    - [2. Post-Operative LOS Model (Dynamic Updates)](#post-operative-los-model-dynamic-updates)
    - [Evaluate](#evaluation-post-operative)
  - [ACTUAL VS PREDICTED](#actual-vs-predicted-los)
  - [Final Summary Evaluation — Length of Stay (LOS) Prediction](#los-final-summary)
- [MODEL 3 - 30-Day Readmission Risk](#model-3-readmission-risk)
  - [top 30 features](#top-30-features-readmission)
  - [Train Test Split](#train-test-split-readmission)
  - [Evaluation](#evaluation-readmission)
  - [Decision Analysis](#decision-analysis-readmission)
  - [Readmission Model — Final Summary](#readmission-model-final-summary)
- [Task 4: Clinical Intervention Recommendations](#task-4-clinical-intervention-recommendations)
  - [Summary of Task 4 Recommendations](#summary-of-task-4-recommendations)

## <a id="architecture"></a> Architecture



**Objective:** Multi-task ML system to predict:

- **Task 1** — Daily discharge readiness (binary classification)
- **Task 2** — Total length of stay (regression)
- **Task 3** — 30-day readmission risk (binary classification)
- **Task 4** — Intervention recommendation (multi-label / rule-augmented)

**Database:** `surgical_predictions.db`  
**Tables:** `patients`, `surgical_encounters`, `daily_clinical_status`, `outcomes`, `hospital_operations_daily`

## <a id="environment-imports"></a> 0. Environment & Imports

In [ ]:
# !pip install --upgrade numpy pandas scipy scikit-learn
# !pip install numpy==1.26.4 matplotlib==3.8.4
# !pip install lightgbm
# !pip install --upgrade pandas lightgbm

In [ ]:
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# from sklearn.ensemble import RandomForestClassifier
# from sklearn.ensemble import RandomForestRegressor
# from lightgbm import LGBMClassifier
# from lightgbm import LGBMRegressor

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    GradientBoostingRegressor, RandomForestRegressor
)
import xgboost as xgb
import shap

from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error,
    roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.calibration import CalibratedClassifierCV
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## <a id="database-connection"></a> Database Connection

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/drive/MyDrive/Colab_Projects/Kaizer/surgical_predictions.db")

# Show all tables
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)

print(tables)

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

### <a id="check-data-each-table"></a> Check data of each table

In [ ]:
patients_df = pd.read_sql(f"SELECT * FROM patients", conn)
patients_df.head()

In [ ]:
print("- \n")
surgical_encounters_df = pd.read_sql(f"SELECT * FROM surgical_encounters", conn)
surgical_encounters_df.head()

print("- \n")
daily_clinical_status_df = pd.read_sql(f"SELECT * FROM daily_clinical_status", conn)
daily_clinical_status_df.head()

print("- \n")
outcomes_df = pd.read_sql(f"SELECT * FROM outcomes", conn)
outcomes_df.head()

print("- \n")
hospital_operations_daily_df = pd.read_sql(f"SELECT * FROM hospital_operations_daily", conn)
hospital_operations_daily_df.head()

### <a id="check-columns-each-table"></a> Check Columns of Each Table

In [ ]:
for table in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn)
    print(f"\nTable: {table}")
    print(df.columns.tolist())

In [ ]:
# Print shape of each table
for table in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    print(f"Table Name: {table}")
    print(f"Shape: {df.shape}")
    print("-" * 40)

## <a id="basic-statistics"></a> 2. Basic Statistics

In [ ]:
# ── patients ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("PATIENTS TABLE")
print("=" * 60)
print(patients_df.describe(include="all").T[["count","mean","std","min","max","top","freq"]].dropna(how="all"))
print("\nMissing values:\n", patients_df.isnull().sum()[patients_df.isnull().sum() > 0])

In [ ]:
# patients_df['smoking_status'].value_counts()
# patients_df['smoking_status'].isnull().sum()

In [ ]:
# ── surgical encounters ───────────────────────────────────────────────────────
print("=" * 60)
print("SURGICAL ENCOUNTERS TABLE")
print("=" * 60)
print(surgical_encounters_df.describe(include="all").T[["count","mean","std","min","max"]].dropna(how="all"))
print("\nMissing values:\n", surgical_encounters_df.isnull().sum()[surgical_encounters_df.isnull().sum() > 0])

In [ ]:
# ── daily clinical status ─────────────────────────────────────────────────────
print("=" * 60)
print("DAILY CLINICAL STATUS TABLE")
print("=" * 60)
print(daily_clinical_status_df.describe().T)
print("\nMissing values:\n", daily_clinical_status_df.isnull().sum()[daily_clinical_status_df.isnull().sum() > 0])

In [ ]:
# ── outcomes ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("OUTCOMES TABLE")
print("=" * 60)
print(outcomes_df.describe(include="all").T[["count","mean","std","min","max"]].dropna(how="all"))
print("\nRe-admission rate:", outcomes_df["readmission_30d"].mean().round(3))
print("\nIntervention distribution:\n", outcomes_df["intervention_recommendation"].value_counts())

In [ ]:
# ── hospital operations ───────────────────────────────────────────────────────
print("=" * 60)
print("HOSPITAL OPERATIONS DAILY")
print("=" * 60)
print(hospital_operations_daily_df.describe().T)

In [ ]:
def basic_eda(df, name):
    print('\n' + '='*50)
    print(f'Dataset: {name}')
    print('='*50)
    print('\nShape:')
    print(df.shape)
    print('\nColumns:')
    print(df.columns.tolist())
    print('\nMissing Values:')
    print(df.isnull().sum())
    print('\nData Types:')
    print(df.dtypes)
    print('\nDuplicate Rows:')
    print(df.duplicated().sum())

basic_eda(patients_df, 'patients')
basic_eda(surgical_encounters_df, 'surgical_encounters')
basic_eda(daily_clinical_status_df, 'daily_clinical_status')
basic_eda(outcomes_df, 'outcomes')
basic_eda(hospital_operations_daily_df, 'hospital_operations_daily')

### <a id="missing-value-percentage"></a> Missing Value Percentage

In [ ]:
def missing_percentage(df):
    missing = pd.DataFrame({
        'column': df.columns,
        'missing_count': df.isnull().sum().values,
        'missing_pct': (df.isnull().sum().values / len(df)) * 100
    })
    return missing.sort_values(by='missing_pct', ascending=False)

missing_percentage(patients_df)
missing_percentage(surgical_encounters_df)
missing_percentage(daily_clinical_status_df)
missing_percentage(outcomes_df)
missing_percentage(hospital_operations_daily_df)

## <a id="exploratory-data-analysis"></a> 3. Exploratory Data Analysis

In [ ]:
# ── Patient Demographics ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Patient Demographics & Clinical Baseline", fontsize=14, fontweight="bold")

# Age distribution
axes[0,0].hist(patients_df["age"], bins=25, color="#4C72B0", edgecolor="white", linewidth=0.5)
axes[0,0].set_title("Age Distribution")
axes[0,0].set_xlabel("Age (years)")
axes[0,0].set_ylabel("Count")

# BMI distribution
axes[0,1].hist(patients_df["bmi"], bins=25, color="#DD8452", edgecolor="white", linewidth=0.5)
axes[0,1].set_title("BMI Distribution")
axes[0,1].set_xlabel("BMI")

# Gender split
g_counts = patients_df["gender"].value_counts()
axes[0,2].bar(g_counts.index, g_counts.values, color=["#4C72B0","#DD8452","#55A868"])
axes[0,2].set_title("Gender Distribution")
axes[0,2].set_ylabel("Count")

# Smoking status
s_counts = patients_df["smoking_status"].value_counts()
axes[1,0].barh(s_counts.index, s_counts.values, color="#4C72B0")
axes[1,0].set_title("Smoking Status")

# ASA class
asa_counts = patients_df["asa_class"].value_counts().sort_index()
axes[1,1].bar(asa_counts.index.astype(str), asa_counts.values, color="#55A868")
axes[1,1].set_title("ASA Classification")
axes[1,1].set_xlabel("ASA Class")

# Comorbidity index
axes[1,2].hist(patients_df["comorbidity_index"], bins=15, color="#C44E52", edgecolor="white")
axes[1,2].set_title("Comorbidity Index")
axes[1,2].set_xlabel("Index Score")

plt.tight_layout()
plt.savefig("plots/eda_demographics.png", bbox_inches="tight")
plt.show()
print("Demographics plot saved.")

In [ ]:
# ── LOS & Discharge Analysis ─────────────────────────────────────────────────
merged_enc = surgical_encounters_df.merge(outcomes_df, on="encounter_id")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Length of Stay & Discharge Patterns", fontsize=13, fontweight="bold")

# LOS distribution
axes[0].hist(merged_enc["actual_los_days"], bins=30, color="#4C72B0", edgecolor="white")
axes[0].axvline(merged_enc["actual_los_days"].median(), color="red", linestyle="--",
                label=f'Median: {merged_enc["actual_los_days"].median():.1f}d')
axes[0].set_title("Actual Length of Stay")
axes[0].set_xlabel("Days")
axes[0].legend()

# LOS by surgery type
merged_enc.groupby("surgery_type")["actual_los_days"].median().sort_values().plot(
    kind="barh", ax=axes[1], color="#DD8452"
)
axes[1].set_title("Median LOS by Surgery Type")
axes[1].set_xlabel("Median Days")

# LOS by complexity
merged_enc.boxplot(column="actual_los_days", by="surgery_complexity", ax=axes[2])
axes[2].set_title("LOS by Surgery Complexity")
axes[2].set_xlabel("Complexity")
axes[2].set_ylabel("LOS Days")
plt.sca(axes[2])
plt.title("LOS by Surgery Complexity")
plt.suptitle("")

plt.tight_layout()
plt.savefig("eda_los.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Readmission Risk Analysis ─────────────────────────────────────────────────
merged_full = patients_df.merge(surgical_encounters_df, on="patient_id").merge(outcomes_df, on="encounter_id")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("30-Day Readmission Risk Drivers", fontsize=13, fontweight="bold")

readm_asa = merged_full.groupby("asa_class")["readmission_30d"].mean()
axes[0,0].bar(readm_asa.index.astype(str), readm_asa.values, color="#C44E52")
axes[0,0].set_title("Readmission Rate by ASA Class")
axes[0,0].set_ylabel("Rate")
axes[0,0].set_xlabel("ASA Class")

readm_comp = merged_full.groupby("surgery_complexity")["readmission_30d"].mean().sort_values()
axes[0,1].barh(readm_comp.index, readm_comp.values, color="#C44E52")
axes[0,1].set_title("Readmission Rate by Complexity")
axes[0,1].set_xlabel("Rate")

merged_full[merged_full["readmission_30d"]==0]["age"].hist(
    bins=20, ax=axes[0,2], alpha=0.6, color="#4C72B0", label="No Readmit")
merged_full[merged_full["readmission_30d"]==1]["age"].hist(
    bins=20, ax=axes[0,2], alpha=0.6, color="#C44E52", label="Readmit")
axes[0,2].set_title("Age: Readmit vs No Readmit")
axes[0,2].legend()

social_readm = merged_full.groupby("social_barrier_flag")["readmission_30d"].mean()
axes[1,0].bar(["No Barrier","Has Barrier"], social_readm.values, color=["#55A868","#C44E52"])
axes[1,0].set_title("Readmission by Social Barrier")
axes[1,0].set_ylabel("Rate")

axes[1,1].boxplot(
    [merged_full[merged_full["readmission_30d"]==0]["complications_count"],
     merged_full[merged_full["readmission_30d"]==1]["complications_count"]],
    labels=["No Readmit","Readmit"]
)
axes[1,1].set_title("Complications Count vs Readmission")
axes[1,1].set_ylabel("Complications")

readm_reasons = outcomes_df[outcomes_df["readmission_30d"]==1]["readmission_reason"].value_counts()
axes[1,2].barh(readm_reasons.index, readm_reasons.values, color="#DD8452")
axes[1,2].set_title("Readmission Reasons")

plt.tight_layout()
plt.savefig("eda_readmission.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Daily Clinical Trajectory Analysis ────────────────────────────────────────
daily_merged = daily_clinical_status_df.merge(
    surgical_encounters_df[["encounter_id","patient_id"]], on="encounter_id"
).merge(outcomes_df[["encounter_id","readmission_30d","actual_los_days"]], on="encounter_id")

traj_cols = ["vitals_stability_score","pain_score","mobility_score",
             "labs_normalized_score","wound_healing_score"]
traj = daily_merged[daily_merged["postop_day"] <= 10].groupby("postop_day")[traj_cols].mean()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B3"]
for col, c in zip(traj_cols, colors):
    ax.plot(traj.index, traj[col], marker="o", label=col.replace("_score","").replace("_"," ").title(), color=c)

ax.set_title("Average Clinical Score Trajectory (Post-Op Days 0–10)", fontsize=13, fontweight="bold")
ax.set_xlabel("Post-Op Day")
ax.set_ylabel("Score")
ax.legend(loc="center right")
plt.tight_layout()
plt.savefig("eda_trajectory.png", bbox_inches="tight")
plt.show()

print("Discharge-ready label distribution by postop day:")
print(daily_clinical_status_df.groupby("postop_day")["discharge_ready_label"].mean())

In [ ]:
# ── Hospital Operations ───────────────────────────────────────────────────────
hospital_operations_daily_df["operation_date"] = pd.to_datetime(hospital_operations_daily_df["operation_date"])
ops_daily_sorted = hospital_operations_daily_df.sort_values("operation_date")

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle("Hospital Operational Trends Over Time", fontsize=13, fontweight="bold")

axes[0].plot(ops_daily_sorted["operation_date"], ops_daily_sorted["bed_occupancy_pct"],
             color="#4C72B0", linewidth=1.2)
axes[0].axhline(ops_daily_sorted["bed_occupancy_pct"].mean(), color="red",
                linestyle="--", alpha=0.7, label=f'Mean: {ops_daily_sorted["bed_occupancy_pct"].mean():.1f}%')
axes[0].set_ylabel("Bed Occupancy %")
axes[0].legend()

axes[1].plot(ops_daily_sorted["operation_date"], ops_daily_sorted["nurse_to_patient_ratio"],
             color="#55A868", linewidth=1.2)
axes[1].set_ylabel("Nurse : Patient Ratio")

axes[2].plot(ops_daily_sorted["operation_date"], ops_daily_sorted["ed_boarding_patients"],
             color="#C44E52", linewidth=1.2)
axes[2].set_ylabel("ED Boarding Patients")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.savefig("eda_operations.png", bbox_inches="tight")
plt.show()

## <a id="merge-tables"></a> Merge Tables

In [ ]:
master_df = daily_clinical_status_df.merge(
    surgical_encounters_df,
    on='encounter_id',
    how='left'
)
master_df = master_df.merge(
    patients_df,
    on='patient_id',
    how='left'
)
master_df = master_df.merge(
    outcomes_df,
    on='encounter_id',
    how='left'
)
master_df.shape
list(master_df.columns)

### <a id="date-conversion"></a> Date Conversion

In [ ]:
master_df['status_date'] = pd.to_datetime(master_df['status_date'])
master_df['admission_date'] = pd.to_datetime(master_df['admission_date'])
master_df['surgery_date'] = pd.to_datetime(master_df['surgery_date'])
hospital_operations_daily_df['operation_date'] = pd.to_datetime(
    hospital_operations_daily_df['operation_date']
)

### <a id="join-operational-table"></a> Join Operational Table

In [ ]:
master_df = master_df.merge(
    hospital_operations_daily_df,
    left_on='status_date',
    right_on='operation_date',
    how='left'
)
master_df.shape

### <a id="feature-engineering"></a> Feature Engineering

In [ ]:
master_df['length_from_admission'] = (
    master_df['status_date'] - master_df['admission_date']
).dt.days

master_df['days_after_surgery'] = (
    master_df['status_date'] - master_df['surgery_date']
).dt.days

master_df['is_weekend'] = master_df['status_date'].dt.weekday.isin([5,6]).astype(int)

master_df['pain_mobility_ratio'] = (
    master_df['pain_score'] / (master_df['mobility_score'] + 1)
)

master_df['recovery_score'] = (
    master_df['mobility_score'] +
    master_df['vitals_stability_score'] +
    master_df['labs_normalized_score'] +
    master_df['wound_healing_score']
)

date_cols = [
    'status_date',
    'admission_date',
    'surgery_date',
    'operation_date'
]

In [ ]:
master_df[date_cols]
master_df.drop(
    columns=date_cols,
    inplace=True
)

### <a id="high-cardinality-features"></a> Identify High Cardinality Features

In [ ]:
for col in master_df.columns:
    if master_df[col].dtype == 'object' or master_df[col].nunique() > 100:
        print(f"Column '{col}': {master_df[col].nunique()} unique values")

In [ ]:
high_cardinality_cols = [
    'status_id',
    # 'encounter_id',
    'mrn',
    'full_name',
    'readmission_reason',
    'intervention_recommendation',
    'patient_id',
    'icu_required',
    # 'readmission_30d',
    'days_after_surgery',
    'is_weekend'
]
master_df.drop(
    columns=high_cardinality_cols,
    inplace=True
)
print(master_df.shape)
master_df.head()

In [ ]:
v

### <a id="encoding"></a> Encoding

In [ ]:
cat_cols = master_df.select_dtypes(include='object').columns
cat_cols

cat_cols = master_df.select_dtypes(include='object').columns
encoded_df = pd.get_dummies(master_df[cat_cols], drop_first=False)

# Combine back with numerical columns
master_df = pd.concat([master_df.drop(columns=cat_cols), encoded_df], axis=1)

In [ ]:
master_df.shape

In [ ]:
master_df.head()

### <a id="fill-missing-values"></a> Fill Missing Values

In [ ]:
for col in master_df.columns:
    if master_df[col].dtype != 'object':
        master_df[col] = master_df[col].fillna(master_df[col].median())

In [ ]:
numeric_cols = [
    'vitals_stability_score', 'pain_score', 'mobility_score',
    'labs_normalized_score', 'wound_healing_score'
]
master_df[numeric_cols] = master_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
master_df_2 = master_df.copy()

In [ ]:
drop_cols=['recovery_score',
    'discharge_ready_day',
    'actual_los_days',
    'length_from_admission',
    'vitals_stability_score',
    'care_plan_complete',
    'postop_day']
master_df = master_df.drop(drop_cols, axis=1)

## <a id="model-building"></a> Model Building

### <a id="model-1-discharge-readiness"></a> MODEL 1 - DISCHARGE READINESS

In [ ]:
master_df

In [ ]:
master_df['discharge_ready_label'].value_counts(normalize=True)

In [ ]:


discharge_model = master_df.copy()

X = discharge_model.drop(
    columns=['discharge_ready_label']
)
y = discharge_model['discharge_ready_label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_train.shape
X_test.shape
y_train.shape
y_test.shape

In [ ]:
from lightgbm import LGBMClassifier

# model = LGBMClassifier(
#     n_estimators=200,
#     learning_rate=0.05,
#     max_depth=10,
#     random_state=42
# )
# model.fit(X_train, y_train)

model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# from xgboost import XGBClassifier
# model = XGBClassifier(
#     n_estimators=200,
#     learning_rate=0.05,
#     max_depth=6,
#     random_state=42,
#     eval_metric='logloss'
# )

model.fit(X_train, y_train)

### <a id="calibrated-model"></a> Calibrated Model

In [ ]:
cal_model = CalibratedClassifierCV(model, method='isotonic')
cal_model.fit(X_train, y_train)

# Raw model probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

# If using calibrated model
y_calibrated = cal_model.predict_proba(X_test)[:, 1]

# For original model
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)

# For calibrated model
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_calibrated, n_bins=10)

plt.figure(figsize=(8, 6))
# Perfect calibration line
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect Calibration')
# Original model
plt.plot(prob_pred, prob_true, marker='o', label='Before Calibration')
# Calibrated model
plt.plot(prob_pred_cal, prob_true_cal, marker='o', label='After Calibration')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual Probability')
plt.title('Calibration Curve (Discharge Readiness Model)')
plt.legend()
plt.grid()
plt.show()

### <a id="prediction-evaluation"></a> Prediction & Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# Probabilities
y_prob = model.predict_proba(X_test)[:, 1]
y_calibrated = cal_model.predict_proba(X_test)[:, 1]

# Use calibrated probs for classification
y_pred = (y_calibrated >= 0.5).astype(int)

print("TASK A — Discharge Readiness")
print(f"AUROC : {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUPRC : {average_precision_score(y_test, y_prob):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Not Ready", "Ready"]))

print(f"\nCalibrated AUROC : {roc_auc_score(y_test, y_calibrated):.4f}")
print(f"Calibrated AUPRC : {average_precision_score(y_test, y_calibrated):.4f}")

### <a id="precision-recall-analysis"></a> Precision-Recall Analysis

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_calibrated)

target_precision = 0.98
for p, r, t in zip(precision, recall, thresholds):
    if p >= target_precision:
        print("\nPR-Based Threshold Insight:")
        print("Threshold:", round(t, 3))
        print("Precision:", round(p, 3), "Recall:", round(r, 3))
        break

### <a id="final-decision-logic"></a> FINAL Decision Logic

In [ ]:
def discharge_decision(prob):
    # High safety threshold (critical)
    if prob >= 0.85:
        return "Discharge"

    # Review zone (clinical judgment)
    elif prob >= 0.5:
        return "Review"

    # Not ready
    else:
        return "Not Ready"

test_df = X_test.copy()
test_df['probability'] = y_calibrated
test_df['decision'] = test_df['probability'].apply(discharge_decision)

### <a id="final-output-analysis"></a> Final Output & Analysis

In [ ]:
# Sample output
print(test_df[['probability', 'decision']].head())

# Decision distribution
print("\nDecision Distribution:")
print(test_df['decision'].value_counts())

# Confusion by decision
print("\nDecision vs Actual:")
print(pd.crosstab(test_df['decision'], y_test))

In [ ]:
import seaborn as sns

cm_df = pd.crosstab(
    test_df['decision'],
    y_test
)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title("Decision vs Actual")
plt.xlabel("Actual (0 = Not Ready, 1 = Ready)")
plt.ylabel("Decision")
plt.show()

In [ ]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

# Use calibrated probabilities (VERY IMPORTANT)
probs = y_calibrated

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Model Evaluation — ROC & Precision-Recall Curves",
    fontsize=14,
    fontweight="bold"
)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, probs)
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y_test, probs):.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curve — Discharge Readiness")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, probs)
axes[1].plot(recall, precision, lw=2, label=f"AUPRC = {average_precision_score(y_test, probs):.3f}")
baseline = y_test.mean()
axes[1].axhline(baseline, color="gray", linestyle="--", label=f"Baseline = {baseline:.2f}")
axes[1].set_title("Precision-Recall Curve — Discharge Readiness")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
importance_df = importance_df.sort_values(by='importance', ascending=False)
importance_df

### <a id="shap-explanation-discharge-readiness"></a> SHAP Explanation for Discharge Readiness

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

## <a id="model-2-los-prediction"></a> MODEL 2 - LOS Prediction

### <a id="admission-los-model"></a> 1. Admission LOS Model

In [ ]:
los_adm_df = master_df_2.drop_duplicates(subset=['encounter_id']).copy()

leakage_cols = [
    'discharge_ready_day',
    'length_from_admission',
    'recovery_score'
]
los_adm_df = los_adm_df.drop(columns=leakage_cols, errors='ignore')

target = 'actual_los_days'
X_adm = los_adm_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y_adm = los_adm_df[target]

print(X_adm.shape, y_adm.shape)

#### <a id="train-model-admission"></a> Train Model

In [ ]:
from lightgbm import LGBMRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X_adm, y_adm, test_size=0.3, random_state=42
)

model_adm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
model_adm.fit(X_train, y_train)

#### <a id="evaluation-admission"></a> Evaluation

In [ ]:
y_pred = model_adm.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("ADMISSION LOS MODEL")
print(f"MAE  : {mae:.3f} days")
print(f"RMSE : {rmse:.3f} days")
print(f"\nMean Actual LOS    : {y_test.mean():.2f}")
print(f"Mean Predicted LOS : {y_pred.mean():.2f}")

### <a id="part-2-post-operative-model"></a> PART 2 POST-OPERATIVE MODEL

In [ ]:
los_dyn_df = master_df_2.copy()
los_dyn_df = los_dyn_df.drop(
    columns=[
        'discharge_ready_day',
        'length_from_admission',
        'recovery_score'
    ],
    errors='ignore'
)

target = 'actual_los_days'
X_dyn = los_dyn_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y_dyn = los_dyn_df[target]

print(X_dyn.shape, y_dyn.shape)

In [ ]:
X_train_dyn, X_test_dyn, y_train_dyn, y_test_dyn = train_test_split(
    X_dyn, y_dyn, test_size=0.3, random_state=42
)

model_dyn = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)
model_dyn.fit(X_train_dyn, y_train_dyn)

#### <a id="evaluation-post-operative"></a> Evaluate

In [ ]:
y_pred_dyn = model_dyn.predict(X_test_dyn)
mae_dyn = mean_absolute_error(y_test_dyn, y_pred_dyn)
rmse_dyn = np.sqrt(mean_squared_error(y_test_dyn, y_pred_dyn))

print("POST-OPERATIVE LOS MODEL")
print(f"MAE  : {mae_dyn:.3f} days")
print(f"RMSE : {rmse_dyn:.3f} days")
print(f"\nMean Actual LOS    : {y_test_dyn.mean():.2f}")
print(f"Mean Predicted LOS : {y_pred_dyn.mean():.2f}")

In [ ]:
feature_imp = pd.DataFrame({
    'feature': X_dyn.columns,
    'importance': model_dyn.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15])
plt.gca().invert_yaxis()
plt.title("Top Features for LOS Prediction (Dynamic Model)")
plt.show()

In [ ]:
residuals = y_test_dyn - y_pred_dyn
plt.figure(figsize=(6,4))
plt.hist(residuals, bins=30)
plt.title("Residual Distribution")
plt.xlabel("Error (days)")
plt.ylabel("Frequency")
plt.show()

### <a id="actual-vs-predicted-los"></a> ACTUAL VS PREDICTED

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test_dyn, y_pred_dyn, alpha=0.5)
plt.plot(
    [y_test_dyn.min(), y_test_dyn.max()],
    [y_test_dyn.min(), y_test_dyn.max()],
    color='red'
)
plt.xlabel("Actual LOS")
plt.ylabel("Predicted LOS")
plt.title("Actual vs Predicted LOS (Dynamic)")
plt.show()

### <a id="los-final-summary"></a> Final Summary Evaluation — Length of Stay (LOS) Prediction

## <a id="model-3-readmission-risk"></a> MODEL 3 - 30-Day Readmission Risk

In [ ]:
# Encounter-level dataset
readmit_df = master_df_2.drop_duplicates(subset=['encounter_id']).copy()
print(readmit_df.shape)
readmit_df['readmission_30d'].value_counts()

In [ ]:
drop_cols = [
    # 'postop_day',
    # 'vitals_stability_score',
    'discharge_ready_day',
    'length_from_admission',
    'recovery_score'
]
readmit_df = readmit_df.drop(columns=drop_cols, errors='ignore')
readmit_df['readmission_30d'].value_counts(normalize=True)

In [ ]:
### <a id="top-30-features-readmission"></a> top 30 features

In [ ]:
target = 'readmission_30d'
X = readmit_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y = readmit_df[target]

print(X.shape, y.shape)
print("Readmission Rate:", y.mean())

#### <a id="train-test-split-readmission"></a> Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
# pip install XGBOOST
readmit_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    random_state=42
)



readmit_model.fit(X_train, y_train)

y_prob_readmit = readmit_model.predict_proba(X_test)[:, 1]

threshold = 0.30
y_pred_readmit = (y_prob_readmit >= threshold).astype(int)

#### <a id="evaluation-readmission"></a> Evaluation

In [ ]:
print("TASK C — READMISSION RISK")
print(f"AUROC : {roc_auc_score(y_test, y_prob_readmit):.4f}")
print(f"AUPRC : {average_precision_score(y_test, y_prob_readmit):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_readmit, target_names=['No Readmit', 'Readmit']))

In [ ]:
#### <a id="decision-analysis-readmission"></a> Decision Analysis

In [ ]:
# Confusion Matrix for Model 3
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_readmit)
).plot(
    ax=ax,
    colorbar=False,
    cmap="Reds"
)
ax.set_title("30-Day Readmission Risk")
plt.tight_layout()
plt.show()

In [ ]:
  def readmission_risk(prob):
      if prob >= 0.45:
          return "High Risk"
      elif prob >= 0.30:
          return "Medium Risk"
      else:
          return "Low Risk"

  test_df = X_test.copy()
  test_df['probability'] = y_prob_readmit
  test_df['risk_group'] = test_df['probability'].apply(readmission_risk)
  print(test_df['risk_group'].value_counts())

In [ ]:
pd.crosstab(test_df['risk_group'], y_test)

In [ ]:
import shap
explainer = shap.TreeExplainer(readmit_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
feature_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': readmit_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15])
plt.gca().invert_yaxis()
plt.title("Top Features for Readmission Risk")
plt.show()

In [ ]:
feature_imp

In [ ]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

# ROC + Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Readmission Risk — ROC & Precision-Recall",
    fontsize=13,
    fontweight="bold"
)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_readmit)
axes[0].plot(fpr, tpr, lw=2, color='red', label=f"AUC = {roc_auc_score(y_test, y_prob_readmit):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob_readmit)
axes[1].plot(recall, precision, lw=2, color='red', label=f"AUPRC = {average_precision_score(y_test, y_prob_readmit):.3f}")
axes[1].axhline(y_test.mean(), color="k", linestyle="--", label=f"Baseline = {y_test.mean():.2f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Feature Importance for Readmission Model
readmit_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': readmit_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=readmit_importance.head(15),
    x='importance',
    y='feature',
    palette='Reds_r'
)
plt.title('Top 15 Drivers of 30-Day Readmission Risk', fontsize=13, fontweight='bold')
plt.xlabel('XGBoost Feature Importance')
plt.ylabel('Clinical/Operational Feature')
plt.tight_layout()
plt.show()

### <a id="readmission-model-final-summary"></a> Readmission Model — Final Summary

## <a id="task-4-clinical-intervention-recommendations"></a> Task 4: Clinical Intervention Recommendations

### <a id="summary-of-task-4-recommendations"></a> Summary of Task 4 Recommendations

In [ ]:
all_rec_list = []
for recs in recommendation_view['recommended_interventions']:
    all_rec_list.extend(recs.split(" | "))

rec_counts = pd.Series(all_rec_list).value_counts().sort_values(ascending=True)

# Plotting using horizontal bar chart
plt.figure(figsize=(10, 6))
rec_counts.plot(kind='barh', color='teal')

plt.title("Frequency of Clinical Recommendations (Test Set)", fontsize=14, fontweight='bold')
plt.xlabel("Count of Recommendations")
plt.ylabel("Intervention Type")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
def generate_recommendations(row):
    recommendations = []

    # 1. High Readmission Risk Intervention
    if row.get('readmission_prob', 0) > 0.4:
        recommendations.append("High-Touch Post-Discharge Follow-up (72 hrs)")
        if row.get('social_barrier_flag') == 1:
            recommendations.append("Social Work / Care Coordination Referral")

    # 2. LOS Reduction (Delayed Discharge Readiness)
    if row.get('discharge_ready_prob', 1) < 0.5:
        if row.get('mobility_score', 100) < 50:
            recommendations.append("Escalate Physical Therapy / Mobility Goals")
        if row.get('pain_score', 0) > 6:
            recommendations.append("Multimodal Pain Management Review")
        if row.get('care_plan_complete') == 0:
            recommendations.append("Prioritize Clinical Care Plan Completion")

    # 3. Operational Quality of Care
    icu = row.get('icu_required', 0)
    vitals = row.get('vitals_stability_score', 100)
    if icu == 1 and vitals < 70:
        recommendations.append("Critical Care Specialist Consultation")

    if not recommendations:
        recommendations.append("Continue Standard ERAS Protocol")

    return " | ".join(recommendations)


current_readmission_test_indices = X_test.index

discharge_model_excluded_cols = [
    'recovery_score', 'discharge_ready_day', 'actual_los_days',
    'length_from_admission', 'vitals_stability_score',
    'care_plan_complete', 'postop_day', 'discharge_ready_label'
]


full_discharge_X = master_df_2.drop(columns=discharge_model_excluded_cols, errors='ignore')


features_for_prediction_raw = full_discharge_X.loc[current_readmission_test_indices]


train_features = model.feature_name_


features_for_prediction = features_for_prediction_raw.reindex(columns=train_features, fill_value=0)


recommendation_view = master_df_2.loc[current_readmission_test_indices].copy()

recommendation_view['readmission_prob'] = y_prob_readmit


recommendation_view['discharge_ready_prob'] = model.predict_proba(features_for_prediction)[:, 1]


recommendation_view['recommended_interventions'] = recommendation_view.apply(generate_recommendations, axis=1)


display(recommendation_view[['readmission_prob', 'discharge_ready_prob', 'recommended_interventions']].head(10))

In [ ]:
all_rec_list = []
for recs in recommendation_view['recommended_interventions']:
    all_rec_list.extend(recs.split(" | "))

rec_counts = pd.Series(all_rec_list).value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=rec_counts.values, y=rec_counts.index, palette='viridis')
plt.title("Frequency of Clinical Recommendations (Test Set)")
plt.xlabel("Count")
plt.ylabel("Intervention Type")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Select and export the relevant clinical columns for validation
export_cols = [
    'readmission_prob',
    'discharge_ready_prob',
    'recommended_interventions'
]

output_filename = 'surgical_clinical_recommendations.csv'
recommendation_view[export_cols].to_csv(output_filename, index=True)
print(f'Successfully exported recommendations to {output_filename}')

In [ ]:
recommendation_view.head()

In [ ]:
excel_file = '/content/surgical_clinical_recommendations.xlsx'
df_excel = pd.read_excel(excel_file)
display(df_excel.head())